# Local vs Global FW-CADIS Weight Windows

The previous example used FW-CADIS to generate **global** weight windows, which aim for uniform statistics everywhere in the model. Often we only care about one tally, for example the flux in a single detector. In that case **local** variance reduction is more efficient because particles are only pushed towards the region of interest.

OpenMC supports local FW-CADIS through the `targets` attribute of the `WeightWindowGenerator`. The adjoint source is then placed only in the target tallies instead of everywhere.

This example has a neutron source in a room surrounded by a concrete shield, with a detector outside one wall. The wall facing the detector is 100 cm thick and the other three walls are 150 cm thick. Global weight windows work on every part of the shield and spend most of their effort pushing neutrons through the thick walls, where we have no detector. Local weight windows only work on the part of the shield between the source and the detector.

More details are in the OpenMC documentation https://docs.openmc.org/en/latest/usersguide/variance_reduction.html

In [ ]:
import copy
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import openmc

# Setting the cross section path to the correct location in the docker image.
# If you are running this outside the docker image you will have to change this path to your local cross section path.
openmc.config['cross_sections'] = Path.home() / 'nuclear_data' / 'cross_sections.xml'

We make air for the room and the detector, and concrete for the shield.

In [ ]:
mat_air = openmc.Material(name='air')
mat_air.add_element("N", 0.784431)
mat_air.add_element("O", 0.210748)
mat_air.add_element("Ar", 0.0046)
mat_air.set_density("g/cc", 0.001205)

mat_concrete = openmc.Material(name='concrete')
mat_concrete.add_element("H", 0.168759)
mat_concrete.add_element("C", 0.001416)
mat_concrete.add_element("O", 0.562524)
mat_concrete.add_element("Na", 0.011838)
mat_concrete.add_element("Mg", 0.0014)
mat_concrete.add_element("Al", 0.021354)
mat_concrete.add_element("Si", 0.204115)
mat_concrete.add_element("K", 0.005656)
mat_concrete.add_element("Ca", 0.018674)
mat_concrete.add_element("Fe", 0.00426)
mat_concrete.set_density("g/cm3", 2.3)

materials = openmc.Materials([mat_air, mat_concrete])

The geometry is a 100 cm wide room of air inside a concrete shield. The shield is 100 cm thick on the right and 150 cm thick on the other three sides. A 20 cm detector cell of air sits outside the right hand wall. The top and bottom surfaces are reflective to keep the problem two dimensional and quick to run.

In [ ]:
room = openmc.model.RectangularPrism(width=100, height=100)
shield = openmc.model.RectangularPrism(width=350, height=400, origin=(-25, 0))
world = openmc.model.RectangularPrism(width=450, height=500, origin=(-25, 0), boundary_type='vacuum')
detector = openmc.model.RectangularPrism(width=20, height=20, origin=(180, 0))
bottom = openmc.ZPlane(z0=-50, boundary_type='reflective')
top = openmc.ZPlane(z0=50, boundary_type='reflective')

room_cell = openmc.Cell(region=-room & +bottom & -top, fill=mat_air, name='room')
shield_cell = openmc.Cell(region=+room & -shield & +bottom & -top, fill=mat_concrete, name='shield')
detector_cell = openmc.Cell(region=-detector & +bottom & -top, fill=mat_air, name='detector')
outside_cell = openmc.Cell(region=+shield & -world & +detector & +bottom & -top, fill=mat_air, name='outside')

geometry = openmc.Geometry([room_cell, shield_cell, detector_cell, outside_cell])
geometry.plot(basis='xy', color_by='cell', outline=True)

A 14 MeV point source and the settings complete the continuous energy model.

In [ ]:
source = openmc.IndependentSource(
    space=openmc.stats.Point((0, 0, 0)),
    angle=openmc.stats.Isotropic(),
    energy=openmc.stats.Discrete([14.06e6], [1.0]),
)

settings = openmc.Settings()
settings.run_mode = 'fixed source'
settings.source = source
settings.particles = 80
settings.batches = 100
settings.inactive = 50  # random ray needs inactive batches

ce_model = openmc.Model(geometry, materials, settings)

Plotting the model instead of the geometry also shows the source location (red), which is in the centre of the room.

In [ ]:
ce_model.plot(n_samples=1, basis='xy', color_by='cell', outline=True, source_kwargs={'color': 'red'})

As in the previous example we convert a copy of the model to multigroup and random ray, which FW-CADIS uses to find the adjoint flux.

In [ ]:
rr_model = copy.deepcopy(ce_model)
rr_model.convert_to_multigroup(method='stochastic_slab', overwrite_mgxs_library=True, nparticles=2000)
rr_model.convert_to_random_ray()

We make a mesh with 10 cm voxels for the weight windows and the random ray source regions, and two tallies:

- a mesh flux tally to view the statistics across the whole model
- a detector flux tally, which is the tally we actually care about

In [ ]:
mesh = openmc.RegularMesh().from_domain(geometry)
mesh.dimension = (45, 50, 1)

mesh_tally = openmc.Tally(name='mesh flux')
mesh_tally.filters = [openmc.MeshFilter(mesh)]
mesh_tally.scores = ['flux']

detector_tally = openmc.Tally(name='detector flux')
detector_tally.filters = [openmc.CellFilter(detector_cell)]
detector_tally.scores = ['flux']

tallies = openmc.Tallies([mesh_tally, detector_tally])

# the target tallies must be present on the random ray model
rr_model.tallies = tallies
rr_model.settings.random_ray['source_region_meshes'] = [(mesh, [rr_model.geometry.root_universe])]

First we generate **global** weight windows, exactly as in the previous example.

In [ ]:
wwg = openmc.WeightWindowGenerator(method='fw_cadis', mesh=mesh)
rr_model.settings.weight_window_generators = wwg

rr_model.run()
Path('weight_windows.h5').rename('weight_windows_global.h5')

Now we generate **local** weight windows. The only change needed is to set the `targets` attribute to the tally (or tallies) that we want to optimise.

In [ ]:
wwg.targets = openmc.Tallies([detector_tally])

rr_model.run()
Path('weight_windows.h5').rename('weight_windows_local.h5')

Plotting the two sets of weight windows shows the difference. Low values (dark) are where particles get split.

The global weight windows fall through all four walls of the shield. The local weight windows only fall through the wall facing the detector and rise in the other three walls, so neutrons heading away from the detector are rouletted instead of split.

In [ ]:
weight_windows = {
    'global': openmc.WeightWindowsList.from_hdf5('weight_windows_global.h5'),
    'local': openmc.WeightWindowsList.from_hdf5('weight_windows_local.h5'),
}
mesh_extent = mesh.bounding_box.extent['xy']

fig, axes = plt.subplots(ncols=2, figsize=(12, 5.5))
for ax, (name, wws) in zip(axes, weight_windows.items()):
    im = ax.imshow(wws[0].lower_ww_bounds.squeeze().T, origin='lower', extent=mesh_extent, norm=LogNorm())
    plt.colorbar(im, ax=ax)
    ce_model.plot(outline='only', extent=mesh_extent, axes=ax, color_by='cell')
    ax.set_title(f'{name} lower weight window bounds')

Now we run the continuous energy Monte Carlo simulation with each set of weight windows using the same number of source particles.

For each run we print the detector flux, its relative error and the figure of merit (FOM), which is 1 / (relative error² × time). A higher FOM means a more efficient simulation. We also plot the relative error of the mesh flux.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 5.5))

for ax, (name, wws) in zip(axes, weight_windows.items()):
    settings = openmc.Settings()
    settings.run_mode = 'fixed source'
    settings.source = source
    settings.particles = 2000
    settings.batches = 20
    settings.weight_windows = wws
    settings.weight_window_checkpoints = {'collision': True, 'surface': True}

    model = openmc.Model(geometry, materials, settings, tallies)
    sp_filename = model.run(output=False)

    with openmc.StatePoint(sp_filename) as sp:
        detector_result = sp.get_tally(name='detector flux')
        mesh_result = sp.get_tally(name='mesh flux')
        time = sp.runtime['simulation']

    rel_err = detector_result.std_dev.item() / detector_result.mean.item()
    fom = 1 / (rel_err**2 * time)
    print(f'{name}: detector flux={detector_result.mean.item():.3e} rel. error={rel_err:.3f} time={time:.1f}s FOM={fom:.1f}')

    mesh_rel_err = mesh_result.get_reshaped_data(value='rel_err', expand_dims=True).squeeze()
    im = ax.imshow(mesh_rel_err.T, origin='lower', extent=mesh_extent, norm=LogNorm(vmin=0.01, vmax=1))
    plt.colorbar(im, ax=ax)
    ce_model.plot(outline='only', extent=mesh_extent, axes=ax, color_by='cell')
    ax.set_title(f'{name} flux relative error')

The global weight windows give a similar relative error everywhere in the model, including behind the three thick walls. The local weight windows give a lower relative error at the detector in less time, at the cost of poor statistics behind the other walls where we have no interest. The white voxels received no scores at all.

The two detector fluxes are close, and they converge to the same value when more particles are used. The local weight windows have a much higher figure of merit for the detector tally. With this few particles the relative errors are themselves only rough estimates, so the figure of merit will vary from run to run.

**Learning Outcomes**

- Global FW-CADIS aims for uniform statistics across the whole model.
- Local FW-CADIS is enabled by setting `targets` on the `WeightWindowGenerator` and optimises for specific tallies.
- When only a few tallies matter, local weight windows avoid wasting compute on regions that do not contribute to them.